<a href="https://colab.research.google.com/github/AndresVillotaVillota/AndresVillotaVillota-Competencia-Kaggle-2025-2/blob/main/99_modelo_soluci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from google.colab import files
import zipfile
import os

# Subir archivo ZIP desde el PC
uploaded = files.upload()

# Obtener nombre del archivo subido
zip_filename = list(uploaded.keys())[0]

# Carpeta donde se extraerán los archivos
extract_folder = "/content/datos_kaggle"
os.makedirs(extract_folder, exist_ok=True)

# Descomprimir
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

print("Archivos descomprimidos en:", extract_folder)


Saving udea-ai-4-eng-20252-pruebas-saber-pro-colombia.zip to udea-ai-4-eng-20252-pruebas-saber-pro-colombia (1).zip
Archivos descomprimidos en: /content/datos_kaggle


In [10]:
import pandas as pd
import os

# Rutas de los archivos extraídos
train_path = os.path.join(extract_folder, "train.csv")
test_path = os.path.join(extract_folder, "test.csv")

# Cargar datasets
df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print(df_train.shape, df_test.shape)
df_train.head()


(692500, 21) (296786, 20)


,ID,PERIODO_ACADEMICO,E_PRGM_ACADEMICO,E_PRGM_DEPARTAMENTO,E_VALORMATRICULAUNIVERSIDAD,E_HORASSEMANATRABAJA,F_ESTRATOVIVIENDA,F_TIENEINTERNET,F_EDUCACIONPADRE,F_TIENELAVADORA,...,E_PRIVADO_LIBERTAD,E_PAGOMATRICULAPROPIO,F_TIENECOMPUTADOR,F_TIENEINTERNET.1,F_EDUCACIONMADRE,RENDIMIENTO_GLOBAL,INDICADOR_1,INDICADOR_2,INDICADOR_3,INDICADOR_4
0,904256,20212,ENFERMERIA,BOGOTÁ,Entre 5.5 millones y menos de 7 millones,Menos de 10 horas,Estrato 3,Si,Técnica o tecnológica incompleta,Si,...,N,No,Si,Si,Postgrado,medio-alto,0.322,0.208,0.310,0.267
1,645256,20212,DERECHO,ATLANTICO,Entre 2.5 millones y menos de 4 millones,0,Estrato 3,No,Técnica o tecnológica completa,Si,...,N,No,Si,No,Técnica o tecnológica incompleta,bajo,0.311,0.215,0.292,0.264
2,308367,20203,MERCADEO Y PUBLICIDAD,BOGOTÁ,Entre 2.5 millones y menos de 4 millones,Más de 30 horas,Estrato 3,Si,Secundaria (Bachillerato) completa,Si,...,N,No,No,Si,Secundaria (Bachillerato) completa,bajo,0.297,0.214,0.305,0.264
3,470353,20195,ADMINISTRACION DE EMPRESAS,SANTANDER,Entre 4 millones y menos de 5.5 millones,0,Estrato 4,Si,No sabe,Si,...,N,No,Si,Si,Secundaria (Bachillerato) completa,alto,0.485,0.172,0.252,0.190
4,989032,20212,PSICOLOGIA,ANTIOQUIA,Entre 2.5 millones y menos de 4 millones,Entre 21 y 30 horas,Estrato 3,Si,Primaria completa,Si,...,N,No,Si,Si,Primaria completa,medio-bajo,0.316,0.232,0.285,0.294


In [11]:
target_col = "RENDIMIENTO_GLOBAL"

y_train = df_train[target_col]
X_train = df_train.drop(columns=[target_col])
X_test = df_test.copy()


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Identificar columnas categóricas y numéricas
cat_cols = X_train.select_dtypes(include=['object']).columns
num_cols = X_train.select_dtypes(exclude=['object']).columns

# Preprocesamiento
preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

# Modelo (puedes cambiarlo por XGBoost, GradientBoosting, etc.)
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    random_state=42
)

# Pipeline completo
clf = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', model)
])


In [13]:
clf.fit(X_train, y_train)
print("Entrenamiento completado.")


Entrenamiento completado.


In [14]:
preds = clf.predict(X_test)
preds[:10]


array(['alto', 'medio-alto', 'alto', 'bajo', 'medio-bajo', 'bajo', 'alto',
       'alto', 'bajo', 'alto'], dtype=object)

In [15]:
submission = pd.DataFrame({
    "ID": df_test["ID"],   # Asegúrate que exista la columna ID en test.csv
    "RENDIMIENTO_GLOBAL": preds
})

submission_path = "/content/submission.csv"
submission.to_csv(submission_path, index=False)

submission.head()


,ID,RENDIMIENTO_GLOBAL
0,550236,alto
1,98545,medio-alto
2,499179,alto
3,782980,bajo
4,785185,medio-bajo


In [16]:
files.download(submission_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>